# VaR Parametrico com GARCH

O **Value-at-Risk (VaR)** e a medida de risco mais utilizada em financas. Ele responde a pergunta:

> *"Qual e a perda maxima esperada em um horizonte de tempo, dado um nivel de confianca?"*

Formalmente, o VaR ao nivel $\alpha$ e definido como:

$$\text{VaR}_\alpha = -\inf\{x : P(r_t \leq x) \geq \alpha\}$$

Em termos praticos, se $\text{VaR}_{5\%} = -2.1\%$, significa que ha apenas 5% de chance
de observar uma perda superior a 2.1% em um dia.

**Conteudo:**
1. VaR Normal (volatilidade constante)
2. VaR com GARCH (volatilidade condicional)
3. VaR com t-Student (caudas pesadas)
4. EWMA - RiskMetrics
5. Comparacao de metodos
6. Exercicios

**Referencias:**
- Jorion, P. (2006). *Value at Risk*. McGraw-Hill.
- RiskMetrics (1996). *Technical Document*. J.P. Morgan.
- Kupiec, P. (1995). Techniques for Verifying the Accuracy of Risk Measurement Models.
- Christoffersen, P. (1998). Evaluating Interval Forecasts. *International Economic Review*.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from archbox.models import GARCH
from archbox.risk import EWMA

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)

# Carregar dados do S&P 500
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']

print(f"Periodo: {returns.index[0].date()} a {returns.index[-1].date()}")
print(f"Observacoes: {len(returns)}")
print(f"Media: {returns.mean():.6f}")
print(f"Desvio padrao: {returns.std():.6f}")
print(f"Curtose: {returns.kurtosis():.2f}")

## 1. VaR Normal

O metodo mais simples assume que os retornos seguem distribuicao normal:

$$r_t \sim N(\mu, \sigma^2)$$

Neste caso, o VaR ao nivel $\alpha$ e:

$$\text{VaR}_\alpha = \mu + z_\alpha \cdot \sigma$$

onde $z_\alpha = \Phi^{-1}(\alpha)$ e o quantil da normal padrao. Para $\alpha = 5\%$, $z_{0.05} = -1.645$;
para $\alpha = 1\%$, $z_{0.01} = -2.326$.

**Limitacoes**: assume volatilidade constante e distribuicao simetrica (ignora caudas pesadas).

In [ ]:
# TODO: Calcule VaR(95%) e VaR(99%) assumindo distribuicao normal

mu = returns.mean()
sigma = returns.std()

# Quantis da normal padrao
z_95 = stats.norm.ppf(0.05)  # -1.645
z_99 = stats.norm.ppf(0.01)  # -2.326

# VaR Normal (estatico)
var_normal_95 = mu + z_95 * sigma
var_normal_99 = mu + z_99 * sigma

print("=== VaR Normal (volatilidade constante) ===")
print(f"z(5%)  = {z_95:.4f}")
print(f"z(1%)  = {z_99:.4f}")
print("")
print(f"VaR(95%) = {var_normal_95:.6f} ({var_normal_95*100:.3f}%)")
print(f"VaR(99%) = {var_normal_99:.6f} ({var_normal_99*100:.3f}%)")
print("")
print("Interpretacao: com 95% de confianca, a perda diaria nao excede "
      f"{abs(var_normal_95)*100:.3f}%")

# Visualizar VaR no histograma
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(returns, bins=80, density=True, alpha=0.6, color='steelblue', label='Retornos')
ax.axvline(var_normal_95, color='orange', linestyle='--', linewidth=2, label=f'VaR 95% = {var_normal_95:.4f}')
ax.axvline(var_normal_99, color='red', linestyle='--', linewidth=2, label=f'VaR 99% = {var_normal_99:.4f}')
ax.set_xlabel('Retorno')
ax.set_ylabel('Densidade')
ax.set_title('Distribuicao dos Retornos e VaR Normal')
ax.legend()
plt.tight_layout()
plt.show()

## 2. VaR com GARCH

O VaR Normal assume volatilidade constante $\sigma$, mas sabemos que a volatilidade financeira
varia ao longo do tempo (**clusters de volatilidade**). O GARCH(1,1) modela isso:

$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

Com volatilidade condicional $\sigma_t$, o VaR **dinamico** e:

$$\text{VaR}_{\alpha,t} = \mu + z_\alpha \cdot \sigma_t$$

Agora o VaR se adapta ao regime de volatilidade atual: maior em periodos turbulentos,
menor em periodos calmos. Esta e a abordagem padrao em gestao de risco.

In [ ]:
# TODO: Estime GARCH(1,1), calcule VaR dinamico com sigma_t

# Estimar GARCH(1,1) com distribuicao normal
model_garch = GARCH(returns.values, p=1, q=1, mean='constant', dist='normal')
results_garch = model_garch.fit(disp=True)

print(results_garch.summary())
print(f"\nPersistencia: {results_garch.persistence():.4f}")
print(f"Meia-vida: {results_garch.half_life():.1f} dias")

# Volatilidade condicional
sigma_t = results_garch.conditional_volatility
mu_garch = results_garch.params[0]  # constante na media

# VaR dinamico com GARCH
var_garch_95 = mu_garch + z_95 * sigma_t
var_garch_99 = mu_garch + z_99 * sigma_t

# Contar violacoes (retornos abaixo do VaR)
violations_95 = (returns.values < var_garch_95).sum()
violations_99 = (returns.values < var_garch_99).sum()
n = len(returns)

print("\n=== VaR GARCH-Normal ===")
print(f"Violacoes VaR(95%): {violations_95}/{n} = {violations_95/n:.4f} (esperado: 0.05)")
print(f"Violacoes VaR(99%): {violations_99}/{n} = {violations_99/n:.4f} (esperado: 0.01)")

## 3. VaR com t-Student

A distribuicao normal subestima eventos extremos. Retornos financeiros tipicamente
apresentam **caudas pesadas** (curtose > 3). A distribuicao t-Student captura isso:

$$\text{VaR}_{\alpha,t} = \mu + t_\alpha(\nu) \cdot \sigma_t$$

onde $t_\alpha(\nu)$ e o quantil da t-Student com $\nu$ graus de liberdade.
Para $\nu$ pequeno (ex: 5-8), as caudas sao significativamente mais pesadas que a normal,
resultando em VaR mais conservador (maior em valor absoluto).

Note que para a t-Student padronizada (variancia unitaria), usamos:

$$t_\alpha^*(\nu) = t_\alpha(\nu) \cdot \sqrt{\frac{\nu - 2}{\nu}}$$

In [ ]:
# TODO: Estime GARCH com distribuicao t-Student, calcule VaR

# Estimar GARCH(1,1) com distribuicao t-Student
model_t = GARCH(returns.values, p=1, q=1, mean='constant', dist='studentt')
results_t = model_t.fit(disp=True)

print(results_t.summary())

# Extrair graus de liberdade estimados
nu = results_t.params[-1]  # ultimo parametro e nu (DoF)
print(f"\nGraus de liberdade estimados: nu = {nu:.2f}")

# Volatilidade condicional do modelo t-Student
sigma_t_student = results_t.conditional_volatility
mu_t = results_t.params[0]

# Quantis da t-Student padronizada (variancia unitaria)
# Para t-Student padronizada: multiplicar quantil por sqrt((nu-2)/nu)
t_95 = stats.t.ppf(0.05, df=nu) * np.sqrt((nu - 2) / nu)
t_99 = stats.t.ppf(0.01, df=nu) * np.sqrt((nu - 2) / nu)

print(f"\nQuantis t-Student padronizada (nu={nu:.1f}):")
print(f"  t*(5%)  = {t_95:.4f}  (vs normal: {z_95:.4f})")
print(f"  t*(1%)  = {t_99:.4f}  (vs normal: {z_99:.4f})")

# VaR dinamico com t-Student
var_t_95 = mu_t + t_95 * sigma_t_student
var_t_99 = mu_t + t_99 * sigma_t_student

# Contar violacoes
viol_t_95 = (returns.values < var_t_95).sum()
viol_t_99 = (returns.values < var_t_99).sum()

print("\n=== VaR GARCH-t ===")
print(f"Violacoes VaR(95%): {viol_t_95}/{n} = {viol_t_95/n:.4f} (esperado: 0.05)")
print(f"Violacoes VaR(99%): {viol_t_99}/{n} = {viol_t_99/n:.4f} (esperado: 0.01)")

## 4. EWMA - RiskMetrics

O modelo **EWMA** (Exponentially Weighted Moving Average) foi popularizado pelo
**RiskMetrics** do J.P. Morgan (1996). E um caso especial do IGARCH(1,1) com $\omega = 0$:

$$\sigma_t^2 = \lambda \sigma_{t-1}^2 + (1 - \lambda) r_{t-1}^2$$

onde $\lambda$ e o fator de decaimento:
- $\lambda = 0.94$ para dados **diarios** (padrao RiskMetrics)
- $\lambda = 0.97$ para dados **mensais**

**Vantagens**: nao requer estimacao (um unico parametro fixo), facil de implementar.
**Desvantagens**: nao ha reversao a media ($\omega = 0$), volatilidade segue random walk.

In [ ]:
# TODO: Calcule VaR com EWMA (lambda=0.94 para dados diarios)

# Modelo EWMA com lambda = 0.94 (padrao RiskMetrics diario)
ewma_model = EWMA(returns.values, lam=0.94)
ewma_results = ewma_model.fit()

sigma_ewma = ewma_results.conditional_volatility

# VaR com EWMA (assumindo distribuicao normal)
var_ewma_95 = mu + z_95 * sigma_ewma
var_ewma_99 = mu + z_99 * sigma_ewma

# Contar violacoes
viol_ewma_95 = (returns.values < var_ewma_95).sum()
viol_ewma_99 = (returns.values < var_ewma_99).sum()

print("=== VaR EWMA (lambda=0.94) ===")
print(f"Volatilidade media EWMA: {sigma_ewma.mean():.6f}")
print(f"Volatilidade final EWMA: {sigma_ewma[-1]:.6f}")
print("")
print(f"Violacoes VaR(95%): {viol_ewma_95}/{n} = {viol_ewma_95/n:.4f} (esperado: 0.05)")
print(f"Violacoes VaR(99%): {viol_ewma_99}/{n} = {viol_ewma_99/n:.4f} (esperado: 0.01)")

# Comparar volatilidade EWMA vs GARCH
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(returns.index, sigma_t, label='GARCH(1,1)', alpha=0.8)
ax.plot(returns.index, sigma_ewma, label='EWMA (λ=0.94)', alpha=0.8)
ax.set_title('Volatilidade Condicional: GARCH vs EWMA')
ax.set_ylabel('Volatilidade (σ_t)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Comparacao de metodos

Vamos comparar os 4 metodos de VaR no mesmo grafico:

| Metodo | Volatilidade | Distribuicao | Parametros |
|--------|-------------|-------------|------------|
| Normal | Constante ($\sigma$) | Normal | $\mu, \sigma$ |
| GARCH-Normal | Condicional ($\sigma_t$) | Normal | $\omega, \alpha, \beta, \mu$ |
| GARCH-t | Condicional ($\sigma_t$) | t-Student | $\omega, \alpha, \beta, \mu, \nu$ |
| EWMA | Condicional ($\sigma_t$) | Normal | $\lambda = 0.94$ |

Os **pontos de violacao** sao retornos que caem abaixo do VaR. Um bom modelo
deve ter taxa de violacao proxima ao nivel de confianca ($\alpha$).

In [ ]:
# TODO: Plote os 4 VaRs no mesmo grafico com retornos

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- VaR 95% ---
ax = axes[0]
ax.plot(returns.index, returns.values, color='gray', alpha=0.4, linewidth=0.5, label='Retornos')
ax.plot(returns.index, np.full(n, var_normal_95), color='blue', linestyle='-', linewidth=1, label='Normal')
ax.plot(returns.index, var_garch_95, color='green', linewidth=1, label='GARCH-Normal')
ax.plot(returns.index, var_t_95, color='red', linewidth=1, label='GARCH-t')
ax.plot(returns.index, var_ewma_95, color='orange', linewidth=1, label='EWMA')

# Marcar violacoes (usando GARCH-t como referencia)
mask_viol = returns.values < var_t_95
ax.scatter(returns.index[mask_viol], returns.values[mask_viol],
           color='red', s=10, zorder=5, alpha=0.6, label='Violacoes (GARCH-t)')

ax.set_title('VaR 95% - Comparacao de Metodos')
ax.set_ylabel('Retorno / VaR')
ax.legend(loc='lower left', fontsize=8)

# --- VaR 99% ---
ax = axes[1]
ax.plot(returns.index, returns.values, color='gray', alpha=0.4, linewidth=0.5, label='Retornos')
ax.plot(returns.index, np.full(n, var_normal_99), color='blue', linestyle='-', linewidth=1, label='Normal')
ax.plot(returns.index, var_garch_99, color='green', linewidth=1, label='GARCH-Normal')
ax.plot(returns.index, var_t_99, color='red', linewidth=1, label='GARCH-t')
ax.plot(returns.index, var_ewma_99, color='orange', linewidth=1, label='EWMA')

mask_viol_99 = returns.values < var_t_99
ax.scatter(returns.index[mask_viol_99], returns.values[mask_viol_99],
           color='darkred', s=15, zorder=5, alpha=0.8, label='Violacoes (GARCH-t)')

ax.set_title('VaR 99% - Comparacao de Metodos')
ax.set_ylabel('Retorno / VaR')
ax.set_xlabel('Data')
ax.legend(loc='lower left', fontsize=8)

plt.tight_layout()
plt.show()

# Tabela resumo de violacoes
print("=" * 70)
print(f"{'Metodo':<20} {'Viol 95%':>10} {'Taxa':>8} {'Viol 99%':>10} {'Taxa':>8}")
print("=" * 70)
print(f"{'Normal':<20} {(returns.values < var_normal_95).sum():>10} {(returns.values < var_normal_95).mean():>8.4f} "
      f"{(returns.values < var_normal_99).sum():>10} {(returns.values < var_normal_99).mean():>8.4f}")
print(f"{'GARCH-Normal':<20} {violations_95:>10} {violations_95/n:>8.4f} "
      f"{violations_99:>10} {violations_99/n:>8.4f}")
print(f"{'GARCH-t':<20} {viol_t_95:>10} {viol_t_95/n:>8.4f} "
      f"{viol_t_99:>10} {viol_t_99/n:>8.4f}")
print(f"{'EWMA (λ=0.94)':<20} {viol_ewma_95:>10} {viol_ewma_95/n:>8.4f} "
      f"{viol_ewma_99:>10} {viol_ewma_99/n:>8.4f}")
print("=" * 70)
print(f"{'Esperado':<20} {'':>10} {'0.0500':>8} {'':>10} {'0.0100':>8}")

## 6. Exercicios

1. **Ibovespa vs S&P 500**: Calcule VaR para o Ibovespa e compare com o S&P 500.
   Mercados emergentes tendem a ter caudas mais pesadas?

2. **Sensibilidade ao lambda**: Varie $\lambda$ do EWMA entre 0.90 e 0.99 e observe
   como isso afeta o VaR. Qual valor minimiza as violacoes?

3. **VaR de portfolio**: Se voce tem 50% em S&P 500 e 50% em Ibovespa,
   o VaR do portfolio e a media ponderada dos VaRs individuais? Por que nao?

4. **Horizonte multi-dia**: Calcule o VaR para horizonte de 10 dias usando
   a regra da raiz quadrada: $\text{VaR}_{10d} = \text{VaR}_{1d} \cdot \sqrt{10}$.
   Quando essa aproximacao e razoavel?

In [ ]:
# TODO: Calcule VaR para Ibovespa e compare com SP500

# Carregar dados do Ibovespa
data_ibov = pd.read_csv('../data/ibovespa_returns.csv', parse_dates=['date'], index_col='date')
returns_ibov = data_ibov['returns']

# Estimar GARCH(1,1) com t-Student para Ibovespa
model_ibov = GARCH(returns_ibov.values, p=1, q=1, mean='constant', dist='studentt')
results_ibov = model_ibov.fit(disp=False)

sigma_ibov = results_ibov.conditional_volatility
mu_ibov = results_ibov.params[0]
nu_ibov = results_ibov.params[-1]

t_95_ibov = stats.t.ppf(0.05, df=nu_ibov) * np.sqrt((nu_ibov - 2) / nu_ibov)
t_99_ibov = stats.t.ppf(0.01, df=nu_ibov) * np.sqrt((nu_ibov - 2) / nu_ibov)

var_ibov_95 = mu_ibov + t_95_ibov * sigma_ibov
var_ibov_99 = mu_ibov + t_99_ibov * sigma_ibov

# Comparacao
print("=== Comparacao S&P 500 vs Ibovespa (GARCH-t) ===\n")
print(f"{'Metrica':<30} {'S&P 500':>12} {'Ibovespa':>12}")
print("-" * 55)
print(f"{'Vol. media':<30} {sigma_t_student.mean():>12.6f} {sigma_ibov.mean():>12.6f}")
print(f"{'Graus de liberdade (nu)':<30} {nu:>12.2f} {nu_ibov:>12.2f}")
print(f"{'VaR 95% medio':<30} {var_t_95.mean():>12.6f} {var_ibov_95.mean():>12.6f}")
print(f"{'VaR 99% medio':<30} {var_t_99.mean():>12.6f} {var_ibov_99.mean():>12.6f}")
print(f"{'Curtose dos retornos':<30} {returns.kurtosis():>12.2f} {returns_ibov.kurtosis():>12.2f}")

# Grafico comparativo
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, ret, var95, var99, title in [
    (axes[0], returns, var_t_95, var_t_99, 'S&P 500'),
    (axes[1], returns_ibov, var_ibov_95, var_ibov_99, 'Ibovespa')
]:
    ax.plot(ret.index, ret.values, color='gray', alpha=0.4, linewidth=0.5, label='Retornos')
    ax.plot(ret.index, var95, color='orange', linewidth=1, label='VaR 95%')
    ax.plot(ret.index, var99, color='red', linewidth=1, label='VaR 99%')
    mask = ret.values < var99
    ax.scatter(ret.index[mask], ret.values[mask], color='red', s=15, zorder=5, label='Violacoes 99%')
    ax.set_title(f'VaR GARCH-t: {title}')
    ax.set_ylabel('Retorno / VaR')
    ax.legend(loc='lower left', fontsize=8)

plt.tight_layout()
plt.show()